# Prediksi Curah Hujan Multi-Skala Waktu — LSTM v2
**Fokus**: Two-Stage Prediction (Klasifikasi + Regresi), Multi-Skala Waktu (1 Jam, 3 Jam, Harian), Bayesian Optimization (Optuna), Mixed Precision, tf.data Pipeline.

## Tujuan
Notebook ini mengimplementasikan sistem prediksi curah hujan hierarkis dua tahap menggunakan LSTM (Long Short-Term Memory) untuk tiga resolusi temporal secara berurutan:
- **Model A**: Prediksi 1 jam ke depan
- **Model B**: Prediksi 3 jam ke depan  
- **Model C**: Prediksi harian (24 jam) ke depan

Setiap model menggunakan pendekatan dua tahap:
- **Tahap 1**: Klasifikasi kejadian hujan (P(hujan)) — Binary LSTM
- **Tahap 2**: Estimasi jumlah curah hujan (mm) — Regression LSTM

## Fitur Utama
- Mixed Precision Training (float16) untuk efisiensi GPU Kaggle P100
- tf.data pipeline dengan caching dan prefetching
- Arsitektur LSTM dinamis (1-2 layer) ditentukan oleh Optuna
- Probability calibration: Isotonic Regression vs Platt Scaling
- SHAP via Random Forest Surrogate (aman untuk LSTM)
- Permutation Importance

In [ ]:
!pip install --upgrade shap optuna scikit-learn --quiet

import os, sys, random, json, logging, warnings, datetime
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import shap

from sklearn.preprocessing import MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance as sk_permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix, log_loss,
    mean_squared_error, mean_absolute_error, r2_score,
    precision_recall_curve, roc_curve, auc, average_precision_score
)
from sklearn.calibration import calibration_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def is_kaggle():
    return os.path.exists('/kaggle/input')

# ============================================================
# KONFIGURASI UTAMA — Ubah TEST_MODE=False untuk Kaggle
# ============================================================
TEST_MODE = True

# Hyperparameter Model LSTM (Baseline Kokoh)
LSTM_CONFIG = {
    'sequence_length': 24,         # Time steps ke belakang
    'use_bidirectional': True,     # Menggunakan Bidirectional LSTM
    'n_lstm_layers': 2,            # Jumlah layer LSTM (1 atau 2)
    'lstm_units_1': 64,            # Unit LSTM layer 1
    'lstm_units_2': 32,            # Unit LSTM layer 2 (jika 2 layer)
    'dropout_rate': 0.2,           # Dropout rate
    'learning_rate': 0.001,        # Learning rate Adam optimizer
    'epochs_occ': 3 if TEST_MODE else 15,  # Epoch training classifier
    'epochs_reg': 3 if TEST_MODE else 20,  # Epoch training regressor
}

CONFIG = {
    'TEST_MODE': TEST_MODE,
    'TEST_ROWS': 2000,
    'BATCH_SIZE': 128,
    'SEED': 42,
    'TRAIN_YEARS': (2005, 2023),
    'VAL_YEAR': 2024,
    'TEST_YEAR': 2025,
}

# Mixed Precision: Aktifkan untuk GPU Kaggle P100
if is_kaggle():
    mixed_precision.set_global_policy('mixed_float16')
    logger.info("Mixed Precision (float16) diaktifkan")

if is_kaggle():
    OUTPUT_BASE = Path('/kaggle/working/outputs/lstm')
else:
    OUTPUT_BASE = Path(r'D:\Github\Projek_Rainfall\DeepLearning_Meteorologi\outputs\lstm')

for scale in ['1h', '3h', 'daily']:
    for subdir in ['plots', 'metrics', 'shap', 'predictions']:
        (OUTPUT_BASE / scale / subdir).mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['SEED'])
logger.info(f"TensorFlow: {tf.__version__}")
logger.info(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU'))}")
logger.info(f"Mode: {'TEST' if TEST_MODE else 'FULL'} | Output: {OUTPUT_BASE}")


## Pemuatan Data
Memuat data cuaca dari sumber CSV dan memilih kolom yang relevan untuk analisis.

In [ ]:
def get_paths():
    if is_kaggle():
        return Path('/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv')
    return Path(r'D:\Github\Projek_Rainfall\Analisis_Meteorologi\open_meteo_jerukagung\cuaca_jerukagung.csv')

ESSENTIAL_COLS = [
    'rain', 'temperature_2m', 'wet_bulb_temperature_2m', 'relative_humidity_2m',
    'dew_point_2m', 'total_column_integrated_water_vapour', 'surface_pressure',
    'pressure_msl', 'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m',
    'cloud_cover', 'cloud_cover_high', 'cloud_cover_mid', 'cloud_cover_low',
    'boundary_layer_height', 'vapour_pressure_deficit', 'sunshine_duration',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
    'direct_normal_irradiance', 'et0_fao_evapotranspiration'
]

def load_data(filepath):
    logger.info(f"Memuat data dari {filepath}")
    df = pd.read_csv(filepath)
    for col in ['datetime', 'date']:
        if col in df.columns:
            df = df.set_index(col)
            break
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Asia/Jakarta').tz_localize(None)
    df.index.name = 'date'
    df = df.sort_index()
    cols = [c for c in ESSENTIAL_COLS if c in df.columns]
    df = df[cols]
    if 'rain' in df.columns:
        df.loc[df['rain'] < 0, 'rain'] = 0
    if CONFIG['TEST_MODE']:
        df = df.tail(CONFIG['TEST_ROWS'])
        logger.info(f"TEST_MODE: menggunakan {len(df)} baris terakhir")
    logger.info(f"Data dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom | {df.index.min()} s/d {df.index.max()}")
    return df

data_path = get_paths()
df_raw = load_data(data_path)
display(df_raw.head())


## Rekayasa Fitur
Menghasilkan fitur turunan dari data mentah: dekomposisi angin, tren atmosfer, fitur siklik, dan indikator fisika cuaca.

In [ ]:
def generate_features(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # 1. Dekomposisi vektor angin (U dan V)
    if 'wind_speed_10m' in df.columns and 'wind_direction_10m' in df.columns:
        wd_rad = df['wind_direction_10m'] * np.pi / 180.0
        df['wind_u'] = -df['wind_speed_10m'] * np.sin(wd_rad)
        df['wind_v'] = -df['wind_speed_10m'] * np.cos(wd_rad)
        df = df.drop(columns=['wind_speed_10m', 'wind_direction_10m'])

    # 2. Tren atmosfer: laju perubahan 3 jam terakhir
    trend_cols = [c for c in ['temperature_2m', 'relative_humidity_2m',
                               'pressure_msl', 'surface_pressure'] if c in df.columns]
    for col in trend_cols:
        df[f'{col}_change_3h'] = df[col].diff(1)
    if 'wind_u' in df.columns:
        df['wind_u_change_3h'] = df['wind_u'].diff(1)
        df['wind_v_change_3h'] = df['wind_v'].diff(1)

    # 3. Dew Point Depression (indikator kejenuhan udara)
    if 'temperature_2m' in df.columns and 'relative_humidity_2m' in df.columns:
        df['dew_point_depression'] = (100 - df['relative_humidity_2m']) / 5

    # 4. Fitur siklik (jam dan bulan)
    df['hour_sin']  = np.sin(2 * np.pi * df.index.hour / 24.0)
    df['hour_cos']  = np.cos(2 * np.pi * df.index.hour / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12.0)

    df = df.dropna()
    return df

df_feat = generate_features(df_raw)
logger.info(f"Fitur dihasilkan: {df_feat.shape[1]} kolom")
df_feat.info()


## Persiapan Target Multi-Skala
Mendefinisikan target prediksi dan threshold kejadian hujan untuk setiap skala waktu.

In [ ]:
SCALE_CONFIG = {
    '1h': {
        'resample': None,
        'shift': -1,
        'threshold': 0.5,
        'label': 'Prediksi 1 Jam',
        'unit': 'mm/jam',
    },
    '3h': {
        'resample': '3h',
        'shift': -1,
        'threshold': 1.0,
        'label': 'Prediksi 3 Jam',
        'unit': 'mm/3jam',
    },
    'daily': {
        'resample': 'D',
        'shift': -1,
        'threshold': 1.0,
        'label': 'Prediksi Harian',
        'unit': 'mm/hari',
    },
}

def prepare_targets(df_feat, scale):
    scfg = SCALE_CONFIG[scale]
    df = df_feat.copy()

    if scfg['resample'] is not None:
        agg_rules = {c: 'mean' for c in df.columns}
        if 'rain' in df.columns:
            agg_rules['rain'] = 'sum'
        df = df.resample(scfg['resample']).agg(agg_rules).dropna()

    df['target_amount'] = df['rain'].shift(scfg['shift'])
    df = df.dropna()
    thr = scfg['threshold']
    df['target_occurrence'] = (df['target_amount'] >= thr).astype(int)

    occ_rate = df['target_occurrence'].mean()
    logger.info(f"[{scale}] Dataset: {len(df):,} baris | Kejadian hujan: {occ_rate:.1%} | Threshold: >={thr} {scfg['unit']}")
    return df


## Fungsi Builder LSTM
Fungsi untuk:
- `create_sequences()`: Mengubah data tabular ke format sekuens 3D untuk input LSTM
- `make_tf_dataset()`: Membuat tf.data.Dataset dengan caching dan prefetching (efisiensi GPU)
- `build_lstm_model()`: Membangun arsitektur LSTM dinamis (1 atau 2 layer, ditentukan Optuna)
- `get_lstm_callbacks()`: EarlyStopping + ReduceLROnPlateau

In [ ]:
def create_sequences(X_arr, y_arr, time_steps):
    """Mengubah array 2D menjadi sequence 3D untuk input LSTM."""
    Xs, ys = [], []
    for i in range(len(X_arr) - time_steps):
        Xs.append(X_arr[i:(i + time_steps)])
        ys.append(y_arr[i + time_steps])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

def make_tf_dataset(X_seq, y_seq, batch_size, shuffle=False):
    """Membuat tf.data.Dataset dengan caching dan prefetching untuk efisiensi."""
    ds = tf.data.Dataset.from_tensor_slices((X_seq, y_seq))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X_seq), seed=CONFIG['SEED'])
    ds = ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
    return ds

def build_lstm_model(ts, n_features, n_layers, units_1, units_2, dropout_rate, output_activation, use_bidirectional=True, output_dtype='float32'):
    """Membangun model LSTM dengan arsitektur kondisional (1 atau 2 layer)."""
    inputs = layers.Input(shape=(ts, n_features), dtype='float32')
    x = inputs

    if n_layers == 2:
        lstm_1 = layers.LSTM(units_1, return_sequences=True, name='lstm_layer_1')
        if use_bidirectional:
            x = layers.Bidirectional(lstm_1, name='bilstm_layer_1')(x)
        else:
            x = lstm_1(x)
        x = layers.Dropout(dropout_rate, name='dropout_1')(x)
        
        lstm_2 = layers.LSTM(units_2, return_sequences=False, name='lstm_layer_2')
        if use_bidirectional:
            x = layers.Bidirectional(lstm_2, name='bilstm_layer_2')(x)
        else:
            x = lstm_2(x)
    else:
        lstm_1 = layers.LSTM(units_1, return_sequences=False, name='lstm_layer_1')
        if use_bidirectional:
            x = layers.Bidirectional(lstm_1, name='bilstm_layer_1')(x)
        else:
            x = lstm_1(x)

    x = layers.BatchNormalization(name='batch_norm')(x)
    x = layers.Dropout(dropout_rate, name='dropout_out')(x)
    x = layers.Dense(16, activation='relu', name='dense_hidden')(x)

    # Output layer harus float32 untuk stabilitas numerik meski mixed precision aktif
    if output_activation == 'sigmoid':
        outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
    else:
        outputs = layers.Dense(1, activation='linear', dtype='float32', name='output')(x)

    model = keras.Model(inputs, outputs)
    return model

def get_lstm_callbacks():
    """EarlyStopping + ReduceLROnPlateau untuk efisiensi training."""
    return [
        keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True, monitor='val_loss', verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=3, monitor='val_loss', verbose=1, min_lr=1e-6
        )
    ]


## Fungsi Metrik Evaluasi
Fungsi untuk menghitung metrik klasifikasi (CSI, POD, FAR, ETS, HSS, ROC-AUC, dll) dan regresi (RMSE, MAE, NSE, KGE, dll).

In [ ]:
def met_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    hits             = np.sum((y_pred == 1) & (y_true == 1))
    misses           = np.sum((y_pred == 0) & (y_true == 1))
    false_alarms     = np.sum((y_pred == 1) & (y_true == 0))
    correct_negatives = np.sum((y_pred == 0) & (y_true == 0))
    total = hits + misses + false_alarms + correct_negatives
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
    pod = hits / (hits + misses) if (hits + misses) > 0 else 0
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
    hits_r = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
    ets = (hits - hits_r) / (hits + misses + false_alarms - hits_r) if (hits + misses + false_alarms - hits_r) > 0 else 0
    denom = ((hits+misses)*(misses+correct_negatives) + (hits+false_alarms)*(false_alarms+correct_negatives))
    hss = (2*(hits*correct_negatives - misses*false_alarms)) / denom if denom > 0 else 0
    return {'CSI': csi, 'POD': pod, 'FAR': far, 'ETS': ets, 'HSS': hss}

def compute_classification_metrics(y_true, prob_uncal, prob_cal, pred_cal):
    y_true = np.asarray(y_true)
    met = met_metrics(y_true, pred_cal)
    pr, rc, _ = precision_recall_curve(y_true, prob_cal)
    pr_auc = auc(rc, pr)
    return {
        'Accuracy':    accuracy_score(y_true, pred_cal),
        'Precision':   precision_score(y_true, pred_cal, zero_division=0),
        'Recall':      recall_score(y_true, pred_cal, zero_division=0),
        'F1':          f1_score(y_true, pred_cal, zero_division=0),
        'ROC_AUC':     roc_auc_score(y_true, prob_cal) if len(np.unique(y_true)) > 1 else 0,
        'PR_AUC':      pr_auc,
        'Brier_Uncal': brier_score_loss(y_true, prob_uncal),
        'Brier_Cal':   brier_score_loss(y_true, prob_cal),
        **met
    }

def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return {k: np.nan for k in ['MAE','RMSE','R2','NSE','KGE','Bias','Correlation']}
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    nse  = 1 - ss_res / (ss_tot + 1e-10)
    r_p  = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else 0
    alpha = np.std(y_pred) / (np.std(y_true) + 1e-10)
    beta  = np.mean(y_pred) / (np.mean(y_true) + 1e-10)
    kge  = 1 - np.sqrt((r_p-1)**2 + (alpha-1)**2 + (beta-1)**2)
    bias = np.mean(y_pred - y_true)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'NSE': nse, 'KGE': kge, 'Bias': bias, 'Correlation': r_p}


## Fungsi Visualisasi
Fungsi untuk menghasilkan setiap jenis plot secara terpisah dan menyimpannya sebagai file PNG.

In [ ]:
def _add_timestamp(ax):
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    ax.annotate(f'Dibuat: {ts}', xy=(1, 0), xycoords='axes fraction',
                fontsize=7, ha='right', va='bottom', color='gray', style='italic')

def save_fig(fig, path):
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info(f"Plot disimpan: {path}")

def plot_reliability_diagram(y_true, prob_uncal, prob_iso, prob_platt, out_dir, scale):
    fig, ax = plt.subplots(figsize=(7, 6))
    f_iso,   m_iso   = calibration_curve(y_true, prob_iso,   n_bins=10)
    f_platt, m_platt = calibration_curve(y_true, prob_platt, n_bins=10)
    f_uncal, m_uncal = calibration_curve(y_true, prob_uncal, n_bins=10)
    ax.plot([0,1],[0,1], 'k:', label='Kalibrasi Sempurna')
    ax.plot(m_uncal, f_uncal, 's--', alpha=0.6, label='Tidak Terkalibrasi')
    ax.plot(m_iso,   f_iso,   'o-',  label='Isotonic')
    ax.plot(m_platt, f_platt, '^-',  label='Platt Scaling')
    ax.set_xlabel('Probabilitas Prediksi Rata-rata', fontsize=12)
    ax.set_ylabel('Fraksi Kejadian Positif', fontsize=12)
    ax.set_title(f'Diagram Reliabilitas — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'reliability_diagram.png')

def plot_probability_distribution(prob_uncal, prob_iso, prob_platt, out_dir, scale):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(prob_uncal, color='red',   alpha=0.35, label='Tidak Terkalibrasi', kde=True, bins=30, ax=ax)
    sns.histplot(prob_iso,   color='blue',  alpha=0.35, label='Isotonic',           kde=True, bins=30, ax=ax)
    sns.histplot(prob_platt, color='green', alpha=0.35, label='Platt Scaling',      kde=True, bins=30, ax=ax)
    ax.set_xlabel('Probabilitas Hujan', fontsize=12)
    ax.set_ylabel('Frekuensi', fontsize=12)
    ax.set_title(f'Distribusi Probabilitas — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'probability_distribution.png')

def plot_confusion_matrix(y_true, y_pred, out_dir, scale):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Tidak Hujan', 'Hujan'],
                yticklabels=['Tidak Hujan', 'Hujan'])
    ax.set_xlabel('Prediksi', fontsize=12)
    ax.set_ylabel('Aktual', fontsize=12)
    ax.set_title(f'Matriks Kebingungan — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'confusion_matrix.png')

def plot_roc_curve(y_true, prob_cal, out_dir, scale):
    fpr, tpr, _ = roc_curve(y_true, prob_cal)
    roc_auc_val = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'Kurva ROC — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'roc_curve.png')

def plot_precision_recall_curve(y_true, prob_cal, out_dir, scale):
    pr, rc, _ = precision_recall_curve(y_true, prob_cal)
    pr_auc_val = auc(rc, pr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(rc, pr, lw=2, label=f'PR (AUC = {pr_auc_val:.3f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'Kurva Precision-Recall — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'precision_recall_curve.png')

def plot_prediction_vs_observation(y_true, y_pred, out_dir, scale):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    max_val = max(y_true.max(), y_pred.max()) if len(y_true) > 0 else 1
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_true, y_pred, alpha=0.3, s=15)
    ax.plot([0, max_val],[0, max_val], 'r--', label='Ideal (y=x)')
    ax.set_xlabel(f'Observasi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel(f'Prediksi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Prediksi vs Observasi — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage (Hanya Saat Hujan)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'prediction_vs_observation.png')

def plot_residual_distribution(y_true, y_pred, out_dir, scale):
    residuals = np.asarray(y_pred) - np.asarray(y_true)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(residuals, kde=True, bins=30, ax=ax)
    ax.axvline(0, color='r', linestyle='--', label='Bias=0')
    ax.set_xlabel(f'Residu (Prediksi - Observasi) [{SCALE_CONFIG[scale]["unit"]}]', fontsize=12)
    ax.set_ylabel('Frekuensi', fontsize=12)
    ax.set_title(f'Distribusi Residu — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'residual_distribution.png')

def plot_timeseries_prediction(y_true, y_pred, index, out_dir, scale, n_show=200):
    if len(y_true) > n_show:
        y_true = y_true[-n_show:]
        y_pred = y_pred[-n_show:]
        index  = index[-n_show:]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(index, y_true, label='Observasi', linewidth=1.5, color='steelblue')
    ax.plot(index, y_pred, label='Prediksi',  linewidth=1.5, color='orangered', alpha=0.85)
    ax.set_xlabel('Waktu', fontsize=12)
    ax.set_ylabel(f'Curah Hujan ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Deret Waktu Prediksi vs Observasi — {SCALE_CONFIG[scale]["label"]}\nLSTM Two-Stage ({n_show} titik terakhir)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'timeseries_prediction_vs_observation.png')

def plot_permutation_importance(importances, feature_names, out_dir, scale, title_suffix=''):
    idx = np.argsort(importances)[::-1][:20]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(range(len(idx)), importances[idx][::-1])
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx[::-1]], fontsize=9)
    ax.set_xlabel('Penurunan Kinerja Rata-rata (Permutation Importance)', fontsize=11)
    ax.set_title(f'Permutation Importance — {SCALE_CONFIG[scale]["label"]}\nLSTM {title_suffix}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    _add_timestamp(ax)
    save_fig(fig, out_dir / f'permutation_importance_{title_suffix.replace(" ","_").lower()}.png')

def plot_shap_importance(shap_values, feature_names, out_dir, scale, title_suffix=''):
    shap_mean = np.abs(shap_values).mean(axis=0)
    idx = np.argsort(shap_mean)[::-1][:20]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(range(len(idx)), shap_mean[idx][::-1])
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx[::-1]], fontsize=9)
    ax.set_xlabel('Nilai SHAP Rata-rata |SHAP|', fontsize=12)
    ax.set_title(f'Kepentingan Fitur SHAP (RF Surrogate) — {SCALE_CONFIG[scale]["label"]}\nLSTM {title_suffix}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    _add_timestamp(ax)
    save_fig(fig, out_dir / f'shap_importance_{title_suffix.replace(" ","_").lower()}.png')

# --- METEOROLOGICAL VALIDATION PLOTS ADDED BY AGENT ---
def categorize_rainfall(amount, scale):
    amount = np.asarray(amount)
    cats = np.zeros(amount.shape, dtype=object)
    if scale == '1h':
        cats[amount < 0.1] = 'No Rain (<0.1)'
        cats[(amount >= 0.1) & (amount < 2.0)] = 'Light (0.1-2.0)'
        cats[(amount >= 2.0) & (amount < 5.0)] = 'Moderate (2.0-5.0)'
        cats[(amount >= 5.0) & (amount < 10.0)] = 'Heavy (5.0-10.0)'
        cats[amount >= 10.0] = 'Very Heavy (>=10)'
    elif scale == '3h':
        cats[amount < 0.5] = 'No Rain (<0.5)'
        cats[(amount >= 0.5) & (amount < 5.0)] = 'Light (0.5-5.0)'
        cats[(amount >= 5.0) & (amount < 15.0)] = 'Moderate (5.0-15.0)'
        cats[(amount >= 15.0) & (amount < 30.0)] = 'Heavy (15.0-30.0)'
        cats[amount >= 30.0] = 'Very Heavy (>=30)'
    else:  # 'daily'
        cats[amount < 0.5] = 'No Rain (<0.5)'
        cats[(amount >= 0.5) & (amount < 20.0)] = 'Light (0.5-20)'
        cats[(amount >= 20.0) & (amount < 50.0)] = 'Moderate (20-50)'
        cats[(amount >= 50.0) & (amount < 100.0)] = 'Heavy (50-100)'
        cats[amount >= 100.0] = 'Very Heavy (>=100)'
    return cats

def plot_meteorological_confusion_matrix(y_true, y_pred, out_dir, scale):
    y_true_cat = categorize_rainfall(y_true, scale)
    y_pred_cat = categorize_rainfall(y_pred, scale)
    
    no_rain_lbl = 'No Rain (<0.1)' if scale == '1h' else 'No Rain (<0.5)'
    light_lbl = 'Light (0.1-2.0)' if scale == '1h' else 'Light (0.5-5.0)' if scale == '3h' else 'Light (0.5-20)'
    mod_lbl = 'Moderate (2.0-5.0)' if scale == '1h' else 'Moderate (5.0-15.0)' if scale == '3h' else 'Moderate (20-50)'
    heavy_lbl = 'Heavy (5.0-10.0)' if scale == '1h' else 'Heavy (15.0-30.0)' if scale == '3h' else 'Heavy (50-100)'
    vheavy_lbl = 'Very Heavy (>=10)' if scale == '1h' else 'Very Heavy (>=30)' if scale == '3h' else 'Very Heavy (>=100)'
    
    order = [no_rain_lbl, light_lbl, mod_lbl, heavy_lbl, vheavy_lbl]
    
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
    cm = sk_confusion_matrix(y_true_cat, y_pred_cat, labels=order)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=[c.split(' (')[0] for c in order],
                yticklabels=[c.split(' (')[0] for c in order])
    ax.set_xlabel('Prediksi Kategori BMKG', fontsize=12)
    ax.set_ylabel('Aktual Kategori BMKG', fontsize=12)
    ax.set_title(f'Matriks Kebingungan Meteorologi BMKG — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'meteorological_confusion_matrix.png')

def plot_csi_vs_threshold(y_true, y_pred, out_dir, scale):
    if scale == '1h':
        thresholds = [0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
    elif scale == '3h':
        thresholds = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 15.0, 20.0]
    else:  # 'daily'
        thresholds = [0.5, 1.0, 5.0, 10.0, 20.0, 30.0, 50.0, 75.0]
        
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    csi_scores = []
    
    for t in thresholds:
        hit = np.sum((y_true >= t) & (y_pred >= t))
        fa = np.sum((y_true < t) & (y_pred >= t))
        miss = np.sum((y_true >= t) & (y_pred < t))
        denominator = hit + fa + miss
        csi = hit / denominator if denominator > 0 else 0.0
        csi_scores.append(csi)
        
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, csi_scores, 'o-', linewidth=2, color='darkviolet', label='CSI (Threat Score)')
    ax.set_xlabel(f'Threshold Curah Hujan ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel('CSI Score', fontsize=12)
    ax.set_title(f'CSI (Threat Score) vs. Threshold Hujan — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'csi_vs_threshold.png')

def plot_hexbin_prediction_vs_observation(y_true, y_pred, out_dir, scale):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if len(y_true) == 0:
        return
    max_val = max(y_true.max(), y_pred.max()) if len(y_true) > 0 else 1
    
    fig, ax = plt.subplots(figsize=(8, 7))
    hb = ax.hexbin(y_true, y_pred, gridsize=30, cmap='YlOrRd', mincnt=1, bins='log')
    cb = fig.colorbar(hb, ax=ax, label='Log10(Frekuensi)')
    ax.plot([0, max_val], [0, max_val], 'b--', alpha=0.7, label='Ideal (y=x)')
    ax.set_xlabel(f'Observasi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel(f'Prediksi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Kepadatan Prediksi vs Observasi (Hexbin) — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'prediction_vs_observation_hexbin.png')


## Fungsi Ekspor Hasil
Fungsi untuk menyimpan metrik, prediksi, dan hyperparameter ke dalam file CSV dan JSON.

In [ ]:
def save_classification_metrics(metrics_dict, out_dir, scale, model_name='lstm'):
    df = pd.DataFrame([metrics_dict])
    df.insert(0, 'model', model_name)
    df.insert(1, 'scale', scale)
    path = out_dir / 'classification_metrics.csv'
    df.to_csv(path, index=False)
    logger.info(f"Metrik klasifikasi disimpan: {path}")
    return df

def save_regression_metrics(metrics_dict, out_dir, scale, model_name='lstm'):
    df = pd.DataFrame([metrics_dict])
    df.insert(0, 'model', model_name)
    df.insert(1, 'scale', scale)
    path = out_dir / 'regression_metrics.csv'
    df.to_csv(path, index=False)
    logger.info(f"Metrik regresi disimpan: {path}")
    return df

def save_predictions(y_true_occ, prob_cal, pred_occ, y_true_reg, y_pred_reg, index, out_dir):
    df = pd.DataFrame({
        'timestamp': index,
        'y_true_occurrence': y_true_occ,
        'prob_rain_calibrated': prob_cal,
        'pred_occurrence': pred_occ,
    })
    path = out_dir / 'predictions.csv'
    df.to_csv(path, index=False)
    logger.info(f"Prediksi disimpan: {path}")

def save_feature_importance(scores, feature_names, out_dir, source='shap'):
    df = pd.DataFrame({'feature': feature_names, 'importance': scores})
    df = df.sort_values('importance', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    df['source'] = source
    path = out_dir / 'feature_importance.csv'
    df.to_csv(path, index=False)
    logger.info(f"Feature importance disimpan: {path}")
    return df

def save_hyperparams(params_occ, params_reg, out_dir):
    hp = {'stage1_classifier': params_occ, 'stage2_regressor': params_reg}
    path = out_dir / 'best_hyperparameters.json'
    with open(path, 'w') as f:
        json.dump(hp, f, indent=2, default=str)
    logger.info(f"Hyperparameter disimpan: {path}")


## Fungsi Optimasi Bayesian (Optuna) dan Kalibrasi Probabilitas
Fungsi untuk:
- `run_optuna_lstm_occ()`: Mencari hyperparameter terbaik untuk LSTM Classifier (30 trials)
- `run_optuna_lstm_reg()`: Mencari hyperparameter terbaik untuk LSTM Regressor (30 trials)
- `calibrate_probabilities()`: Membandingkan Isotonic Regression vs Platt Scaling, memilih yang terbaik
- `apply_calibrator()`: Menerapkan kalibrasi pada data baru

In [ ]:
def print_callback(study, trial):
    print(f"Trial {trial.number:3d} | Val Score: {trial.value:.5f} | "
          f"Best: {study.best_value:.5f} | Params: {study.best_params}")

def run_optuna_lstm_occ(X_train_s, y_train_occ, X_val_s, y_val_occ, n_trials, n_features):
    """Optuna Stage 1: Mencari hyperparameter terbaik untuk LSTM Classifier."""
    def objective(trial):
        keras.backend.clear_session()
        ts      = trial.suggest_categorical('sequence_length', [8, 16, 24, 32])
        n_lay   = trial.suggest_int('n_lstm_layers', 1, 2)
        u1      = trial.suggest_int('lstm_units_1', 32, 192, step=32)
        u2      = trial.suggest_int('lstm_units_2', 16, 96, step=16)
        dr      = trial.suggest_float('dropout_rate', 0.1, 0.5)
        lr      = trial.suggest_float('learning_rate', 5e-4, 1e-2, log=True)

        Xt, yt = create_sequences(X_train_s, y_train_occ, ts)
        Xv, yv = create_sequences(X_val_s,   y_val_occ,   ts)

        if len(Xt) < 10 or len(Xv) < 5:
            return 1.0

        model = build_lstm_model(ts, n_features, n_lay, u1, u2, dr, 'sigmoid')
        model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy')
        ds_tr = make_tf_dataset(Xt, yt, CONFIG['BATCH_SIZE'], shuffle=False)
        ds_vl = make_tf_dataset(Xv, yv, CONFIG['BATCH_SIZE'], shuffle=False)
        hist  = model.fit(ds_tr, validation_data=ds_vl,
                          epochs=CONFIG['N_EPOCHS_OCC'], verbose=0,
                          callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
        return min(hist.history['val_loss'])

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, callbacks=[print_callback])
    return study.best_params

def run_optuna_lstm_reg(X_train_s, y_train_reg, X_val_s, y_val_reg, n_trials, n_features):
    """Optuna Stage 2: Mencari hyperparameter terbaik untuk LSTM Regressor."""
    def objective(trial):
        keras.backend.clear_session()
        ts      = trial.suggest_categorical('sequence_length', [8, 16, 24, 32])
        n_lay   = trial.suggest_int('n_lstm_layers', 1, 2)
        u1      = trial.suggest_int('lstm_units_1', 32, 192, step=32)
        u2      = trial.suggest_int('lstm_units_2', 16, 96, step=16)
        dr      = trial.suggest_float('dropout_rate', 0.1, 0.5)
        lr      = trial.suggest_float('learning_rate', 5e-4, 1e-2, log=True)

        Xt, yt = create_sequences(X_train_s, y_train_reg, ts)
        Xv, yv = create_sequences(X_val_s,   y_val_reg,   ts)

        rain_tr = yt > 0
        rain_vl = yv > 0
        Xt_r, yt_r = Xt[rain_tr], np.log1p(yt[rain_tr])
        Xv_r, yv_r = Xv[rain_vl], np.log1p(yv[rain_vl])

        if len(Xt_r) < 5 or len(Xv_r) < 3:
            return 99.0

        model = build_lstm_model(ts, n_features, n_lay, u1, u2, dr, 'linear')
        model.compile(optimizer=keras.optimizers.Adam(lr), loss='mae')
        ds_tr = make_tf_dataset(Xt_r, yt_r, CONFIG['BATCH_SIZE'], shuffle=False)
        ds_vl = make_tf_dataset(Xv_r, yv_r, CONFIG['BATCH_SIZE'], shuffle=False)
        hist  = model.fit(ds_tr, validation_data=ds_vl,
                          epochs=CONFIG['N_EPOCHS_REG'], verbose=0,
                          callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
        return min(hist.history['val_loss'])

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, callbacks=[print_callback])
    return study.best_params

def calibrate_probabilities(prob_uncal_val, y_val):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_uncal_val, y_val)
    prob_iso = iso.predict(prob_uncal_val)

    n_classes_val = len(np.unique(y_val))
    if n_classes_val < 2:
        # Tidak bisa Platt jika hanya satu kelas — gunakan Isotonic saja
        logger.warning("Hanya satu kelas di val set — melewati Platt Scaling, gunakan Isotonic")
        prob_platt = prob_iso.copy()
        brier_iso  = brier_score_loss(y_val, prob_iso)
        logger.info(f"Kalibrasi terpilih: Isotonic (Brier: {brier_iso:.4f})")
        return iso, 'Isotonic', prob_iso, prob_platt

    platt = LogisticRegression()
    platt.fit(prob_uncal_val.reshape(-1, 1), y_val)
    prob_platt = platt.predict_proba(prob_uncal_val.reshape(-1, 1))[:, 1]
    brier_iso   = brier_score_loss(y_val, prob_iso)
    brier_platt = brier_score_loss(y_val, prob_platt)
    if brier_iso <= brier_platt:
        best, method = iso, 'Isotonic'
    else:
        best, method = platt, 'Platt Scaling'
    logger.info(f"Kalibrasi terpilih: {method} (Brier: {min(brier_iso, brier_platt):.4f})")
    return best, method, prob_iso, prob_platt

def apply_calibrator(cal, prob):
    if hasattr(cal, 'predict_proba'):
        return cal.predict_proba(prob.reshape(-1, 1))[:, 1]
    return cal.predict(prob)


## Analisis Kepentingan Fitur (SHAP + Permutation Importance)
Fungsi untuk:
- `compute_shap_rf_surrogate()`: Menghitung SHAP menggunakan Random Forest Surrogate (aman untuk LSTM)
- `compute_permutation_importance_lstm()`: Menghitung Permutation Importance langsung pada model LSTM

In [ ]:
def compute_shap_rf_surrogate(X_train, y_train, X_test, feature_names, task='clf', n_sample=500):
    """
    Menghitung SHAP menggunakan Random Forest Surrogate Model.
    Aman untuk LSTM karena tidak memerlukan akses ke model Keras secara langsung.
    """
    logger.info(f"Melatih RF Surrogate untuk SHAP ({task})...")
    # Pastikan input adalah 2D array (n_samples, n_features)
    X_train_2d = X_train.reshape(len(X_train), -1) if X_train.ndim > 2 else X_train
    X_test_2d  = X_test.reshape(len(X_test), -1)  if X_test.ndim > 2 else X_test
    # Ambil hanya n_features pertama jika ada dimensi ekstra
    n_feat = len(feature_names)
    X_train_2d = X_train_2d[:, :n_feat]
    X_test_2d  = X_test_2d[:, :n_feat]

    if task == 'clf':
        surrogate = RandomForestClassifier(n_estimators=100, random_state=CONFIG['SEED'], n_jobs=-1)
        surrogate.fit(X_train_2d, y_train)
    else:
        surrogate = RandomForestRegressor(n_estimators=100, random_state=CONFIG['SEED'], n_jobs=-1)
        surrogate.fit(X_train_2d, y_train)

    # Sample untuk efisiensi
    X_sample = X_test_2d
    if len(X_test_2d) > n_sample:
        idx = np.random.choice(len(X_test_2d), n_sample, replace=False)
        X_sample = X_test_2d[idx]

    explainer = shap.TreeExplainer(surrogate)
    shap_vals = explainer.shap_values(X_sample)

    # Untuk classifier RF, shap_values bisa berupa list [class0, class1] atau ndarray 3D
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]  # Ambil class positif
    elif shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1]  # (n_samples, n_features, n_classes) -> class positif

    return shap_vals, feature_names

def compute_permutation_importance_lstm(model, X_seq, y_true, feature_names, n_repeat=3):
    """
    Menghitung Permutation Importance dengan mengacak satu fitur per kali
    dan mengukur penurunan kinerja model LSTM.
    """
    logger.info("Menghitung Permutation Importance LSTM...")
    base_pred = model.predict(X_seq, verbose=0).ravel()
    base_loss = mean_absolute_error(y_true, base_pred)

    importances = np.zeros(X_seq.shape[2])
    for fi in range(X_seq.shape[2]):
        losses = []
        for _ in range(n_repeat):
            X_perm = X_seq.copy()
            idx = np.random.permutation(X_perm.shape[0])
            X_perm[:, :, fi] = X_perm[idx, :, fi]
            pred_perm = model.predict(X_perm, verbose=0).ravel()
            losses.append(mean_absolute_error(y_true, pred_perm))
        importances[fi] = np.mean(losses) - base_loss

    logger.info("Permutation Importance selesai")
    return importances


## Pipeline Utama LSTM
Fungsi `run_lstm_pipeline(scale, df_feat)` menjalankan pipeline lengkap untuk satu skala waktu:

1. Persiapan target dan pemisahan data kronologis
2. Optimasi Optuna Stage 1 (Klasifikasi)
3. Optimasi Optuna Stage 2 (Regresi)
4. Pelatihan model LSTM akhir dengan tf.data pipeline + EarlyStopping + ReduceLROnPlateau
5. Kalibrasi probabilitas (Isotonic vs Platt)
6. Evaluasi pada data uji
7. SHAP (RF Surrogate) + Permutation Importance
8. Pembuatan semua plot (9 plot terpisah)
9. Ekspor semua file output

In [ ]:
def run_lstm_pipeline(scale, df_feat):
    scfg = SCALE_CONFIG[scale]
    out_scale   = OUTPUT_BASE / scale
    out_plots   = out_scale / 'plots'
    out_metrics = out_scale / 'metrics'
    out_shap    = out_scale / 'shap'
    out_preds   = out_scale / 'predictions'

    logger.info(f"\n{'='*60}")
    logger.info(f"PIPELINE LSTM — {scfg['label'].upper()}")
    logger.info(f"{'='*60}")

    # -------------------------------------------------------
    # 1. Siapkan target
    # -------------------------------------------------------
    df = prepare_targets(df_feat, scale)
    features   = df.drop(columns=['target_amount', 'target_occurrence'])
    targets    = df[['target_amount', 'target_occurrence']]
    feat_names = features.columns.tolist()
    n_features = len(feat_names)

    train_mask = (df.index.year >= CONFIG['TRAIN_YEARS'][0]) & (df.index.year <= CONFIG['TRAIN_YEARS'][1])
    val_mask   = df.index.year == CONFIG['VAL_YEAR']
    test_mask  = df.index.year == CONFIG['TEST_YEAR']

    if CONFIG['TEST_MODE']:
        n  = len(df)
        tr = int(n * 0.70)
        vl = int(n * 0.85)
        train_mask = pd.Series([False]*n, index=df.index)
        val_mask   = pd.Series([False]*n, index=df.index)
        test_mask  = pd.Series([False]*n, index=df.index)
        train_mask.iloc[:tr] = True
        val_mask.iloc[tr:vl]  = True
        test_mask.iloc[vl:]   = True

    if train_mask.sum() < 50 or val_mask.sum() < 20 or test_mask.sum() < 10:
        logger.warning(f"[{scale}] Data tidak cukup — skip pipeline")
        return None

    X_train, y_train = features[train_mask], targets[train_mask]
    X_val,   y_val   = features[val_mask],   targets[val_mask]
    X_test,  y_test  = features[test_mask],  targets[test_mask]

    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s   = scaler.transform(X_val).astype(np.float32)
    X_test_s  = scaler.transform(X_test).astype(np.float32)

    y_train_occ = y_train['target_occurrence'].values.astype(np.float32)
    y_val_occ   = y_val['target_occurrence'].values.astype(np.float32)
    y_test_occ  = y_test['target_occurrence'].values.astype(np.float32)

    y_train_amt = y_train['target_amount'].values.astype(np.float32)
    y_val_amt   = y_val['target_amount'].values.astype(np.float32)
    y_test_amt  = y_test['target_amount'].values.astype(np.float32)

    # -------------------------------------------------------
    # 2. Setup Hyperparameters (Direct Baseline - No Optuna)
    # -------------------------------------------------------
    p_occ = {
        'sequence_length': LSTM_CONFIG['sequence_length'],
        'n_lstm_layers': LSTM_CONFIG['n_lstm_layers'],
        'lstm_units_1': LSTM_CONFIG['lstm_units_1'],
        'lstm_units_2': LSTM_CONFIG['lstm_units_2'],
        'dropout_rate': LSTM_CONFIG['dropout_rate'],
        'learning_rate': LSTM_CONFIG['learning_rate'],
        'use_bidirectional': LSTM_CONFIG.get('use_bidirectional', True),
    }
    p_reg = p_occ.copy()
    logger.info(f"[{scale}] Menggunakan Baseline Hyperparameters: {p_occ}")

    save_hyperparams(p_occ, p_reg, out_preds)

    # -------------------------------------------------------
    # 4. Latih Model Akhir — Stage 1 (Classifier)
    # -------------------------------------------------------
    ts_occ = p_occ['sequence_length']
    keras.backend.clear_session()
    model_occ = build_lstm_model(ts_occ, n_features,
                                  p_occ['n_lstm_layers'], p_occ['lstm_units_1'],
                                  p_occ['lstm_units_2'],  p_occ['dropout_rate'], 'sigmoid',
                                  use_bidirectional=p_occ['use_bidirectional'])
    model_occ.compile(optimizer=keras.optimizers.Adam(p_occ['learning_rate']),
                      loss='binary_crossentropy')

    Xt_occ, yt_occ = create_sequences(X_train_s, y_train_occ, ts_occ)
    Xv_occ, yv_occ = create_sequences(X_val_s,   y_val_occ,   ts_occ)
    Xe_occ, ye_occ = create_sequences(X_test_s,  y_test_occ,  ts_occ)

    ds_tr_occ = make_tf_dataset(Xt_occ, yt_occ, CONFIG['BATCH_SIZE'])
    ds_vl_occ = make_tf_dataset(Xv_occ, yv_occ, CONFIG['BATCH_SIZE'])
    model_occ.fit(ds_tr_occ, validation_data=ds_vl_occ,
                  epochs=LSTM_CONFIG['epochs_occ'], callbacks=get_lstm_callbacks())

    prob_val_uncal  = model_occ.predict(Xv_occ, verbose=0).ravel()
    prob_test_uncal = model_occ.predict(Xe_occ, verbose=0).ravel()

    cal, cal_method, prob_iso_val, prob_platt_val = calibrate_probabilities(prob_val_uncal, yv_occ)
    prob_test_cal = apply_calibrator(cal, prob_test_uncal)
    pred_test_occ = (prob_test_cal >= 0.5).astype(int)

    # -------------------------------------------------------
    # 5. Latih Model Akhir — Stage 2 (Regressor)
    # -------------------------------------------------------
    ts_reg = p_reg['sequence_length']
    keras.backend.clear_session()
    model_reg = build_lstm_model(ts_reg, n_features,
                                  p_reg['n_lstm_layers'], p_reg['lstm_units_1'],
                                  p_reg['lstm_units_2'],  p_reg['dropout_rate'], 'linear',
                                  use_bidirectional=p_reg['use_bidirectional'])
    model_reg.compile(optimizer=keras.optimizers.Adam(p_reg['learning_rate']), loss='mae')

    Xt_reg, yt_reg_full = create_sequences(X_train_s, y_train_amt, ts_reg)
    Xv_reg, yv_reg_full = create_sequences(X_val_s,   y_val_amt,   ts_reg)
    Xe_reg, ye_reg_full = create_sequences(X_test_s,  y_test_amt,  ts_reg)

    # Filter hanya saat hujan
    thr = scfg['threshold']
    rain_tr = yt_reg_full >= thr
    rain_vl = yv_reg_full >= thr
    rain_te = ye_reg_full >= thr

    Xt_r, yt_r = Xt_reg[rain_tr], np.log1p(yt_reg_full[rain_tr])
    Xv_r, yv_r = Xv_reg[rain_vl], np.log1p(yv_reg_full[rain_vl])
    Xe_r       = Xe_reg[rain_te]
    ye_r       = ye_reg_full[rain_te]

    pred_test_reg = np.zeros(rain_te.sum())
    test_reg_index = X_test.index[ts_reg:][rain_te] if len(X_test) > ts_reg else X_test.index[:0]

    if len(Xt_r) > 5 and len(Xv_r) > 3:
        ds_tr_reg = make_tf_dataset(Xt_r, yt_r, CONFIG['BATCH_SIZE'])
        ds_vl_reg = make_tf_dataset(Xv_r, yv_r, CONFIG['BATCH_SIZE'])
        model_reg.fit(ds_tr_reg, validation_data=ds_vl_reg,
                      epochs=LSTM_CONFIG['epochs_reg'], callbacks=get_lstm_callbacks())
        if len(Xe_r) > 0:
            pred_log = model_reg.predict(Xe_r, verbose=0).ravel()
            pred_test_reg = np.maximum(np.expm1(pred_log), 0)

    # -------------------------------------------------------
    # 6. Evaluasi
    # -------------------------------------------------------
    clf_metrics = compute_classification_metrics(ye_occ, prob_test_uncal, prob_test_cal, pred_test_occ)
    reg_metrics = compute_regression_metrics(ye_r, pred_test_reg) if len(ye_r) > 1 else {}
    logger.info(f"[{scale}] Metrik klasifikasi: {clf_metrics}")
    logger.info(f"[{scale}] Metrik regresi: {reg_metrics}")

    save_classification_metrics(clf_metrics, out_metrics, scale)
    if reg_metrics:
        save_regression_metrics(reg_metrics, out_metrics, scale)

    # -------------------------------------------------------
    # 7. Simpan Prediksi
    # -------------------------------------------------------
    save_predictions(ye_occ, prob_test_cal, pred_test_occ,
                     ye_r, pred_test_reg if len(pred_test_reg)>0 else None,
                     X_test.index[ts_occ:].tolist(), out_preds)

    # -------------------------------------------------------
    # 8. Feature Importance: SHAP + Permutation
    # -------------------------------------------------------
    try:
        logger.info(f"[{scale}] Menghitung SHAP (RF Surrogate) Stage 1...")
        shap_vals, shap_feats = compute_shap_rf_surrogate(
            X_train_s, y_train_occ, X_test_s, feat_names, task='clf')
        save_feature_importance(np.abs(shap_vals).mean(axis=0), feat_names, out_shap, 'shap_clf')
        plot_shap_importance(shap_vals, feat_names, out_shap, scale, title_suffix='Stage 1 Classifier')
    except Exception as e:
        logger.warning(f"[{scale}] SHAP error: {e}")

    try:
        logger.info(f"[{scale}] Menghitung Permutation Importance Stage 1...")
        perm_imp = compute_permutation_importance_lstm(
            model_occ, Xe_occ[:min(200, len(Xe_occ))],
            ye_occ[:min(200, len(ye_occ))], feat_names, n_repeat=2)
        plot_permutation_importance(perm_imp, feat_names, out_shap, scale, title_suffix='Stage 1 Classifier')
        # Simpan dengan nama berbeda agar tidak menimpa SHAP feature_importance.csv
        df_perm = pd.DataFrame({'feature': feat_names, 'importance': perm_imp})
        df_perm = df_perm.sort_values('importance', ascending=False).reset_index(drop=True)
        df_perm['rank'] = df_perm.index + 1
        df_perm['source'] = 'permutation'
        df_perm.to_csv(out_shap / 'feature_importance_permutation.csv', index=False)
        logger.info(f"Permutation importance disimpan: {out_shap / 'feature_importance_permutation.csv'}")
    except Exception as e:
        logger.warning(f"[{scale}] Permutation Importance error: {e}")

    if len(Xe_r) > 5:
        try:
            logger.info(f"[{scale}] Menghitung SHAP Stage 2 (Regressor)...")
            shap_reg_vals, _ = compute_shap_rf_surrogate(
                Xt_r.reshape(len(Xt_r), -1)[:, :n_features],
                yt_r, Xe_r.reshape(len(Xe_r), -1)[:, :n_features], feat_names, task='reg')
            plot_shap_importance(shap_reg_vals, feat_names, out_shap, scale, title_suffix='Stage 2 Regressor')
        except Exception as e:
            logger.warning(f"[{scale}] SHAP Regressor error: {e}")

    # -------------------------------------------------------
    # 9. Visualisasi — Section A (Reliabilitas)
    # -------------------------------------------------------
    iso_test   = IsotonicRegression(out_of_bounds='clip').fit(prob_val_uncal, yv_occ)
    platt_test = LogisticRegression().fit(prob_val_uncal.reshape(-1,1), yv_occ)
    prob_iso_test   = iso_test.predict(prob_test_uncal)
    prob_platt_test = platt_test.predict_proba(prob_test_uncal.reshape(-1,1))[:,1]

    plot_reliability_diagram(ye_occ, prob_test_uncal, prob_iso_test, prob_platt_test, out_plots, scale)
    plot_probability_distribution(prob_test_uncal, prob_iso_test, prob_platt_test, out_plots, scale)

    # -------------------------------------------------------
    # 10. Visualisasi — Section B (Klasifikasi)
    # -------------------------------------------------------
    plot_confusion_matrix(ye_occ, pred_test_occ, out_plots, scale)
    if len(np.unique(ye_occ)) > 1:
        plot_roc_curve(ye_occ, prob_test_cal, out_plots, scale)
        plot_precision_recall_curve(ye_occ, prob_test_cal, out_plots, scale)

    # -------------------------------------------------------
    # 11. Visualisasi — Section C (Regresi & Meteorological Validation)
    # -------------------------------------------------------
    if len(ye_r) > 1 and len(pred_test_reg) > 0:
        plot_prediction_vs_observation(ye_r, pred_test_reg, out_plots, scale)
        plot_residual_distribution(ye_r, pred_test_reg, out_plots, scale)
        plot_timeseries_prediction(ye_r, pred_test_reg, test_reg_index.tolist(), out_plots, scale)
        plot_hexbin_prediction_vs_observation(ye_r, pred_test_reg, out_plots, scale)

    # 12. Meteorological Validation (BMKG Category Confusion Matrix & CSI vs Threshold)
    try:
        df_align = pd.DataFrame(index=X_test.index)
        df_align['y_true_amount'] = y_test['target_amount']
        df_align['pred_occ'] = np.nan
        df_align.loc[X_test.index[ts_occ:], 'pred_occ'] = pred_test_occ
        df_align['pred_reg'] = np.nan
        if model_reg is not None and len(pred_test_reg) > 0:
            df_align.loc[test_reg_index, 'pred_reg'] = pred_test_reg
        df_align = df_align.dropna(subset=['pred_occ'])
        df_align['pred_reg'] = df_align['pred_reg'].fillna(0.0)
        df_align['y_pred_amount'] = df_align['pred_reg'] * df_align['pred_occ']
        
        y_true_all = df_align['y_true_amount'].values
        y_pred_all = df_align['y_pred_amount'].values
        
        plot_meteorological_confusion_matrix(y_true_all, y_pred_all, out_plots, scale)
        plot_csi_vs_threshold(y_true_all, y_pred_all, out_plots, scale)
    except Exception as e:
        logger.warning(f"[{scale}] Meteorological validation plotting error: {e}")

    logger.info(f"[{scale}] Pipeline LSTM selesai! Output: {out_scale}")

    return {
        'model': 'lstm',
        'scale': scale,
        **{f'clf_{k}': v for k, v in clf_metrics.items()},
        **{f'reg_{k}': v for k, v in reg_metrics.items()},
    }


## Eksekusi: Pemuatan Data
Memuat dan memproses data cuaca sebelum menjalankan pipeline.

In [ ]:
data_path = get_paths()
df_raw  = load_data(data_path)
df_feat = generate_features(df_raw)
feature_names = df_feat.drop(columns=['rain'], errors='ignore').columns.tolist()
logger.info(f"Fitur tersedia: {len(feature_names)} kolom")
df_feat.info()


## Pipeline Prediksi 1 Jam
Menjalankan pipeline lengkap LSTM untuk prediksi curah hujan 1 jam ke depan.

In [ ]:
results_1h = run_lstm_pipeline('1h', df_feat)
print("\n[SELESAI] Pipeline LSTM 1 Jam")
if results_1h:
    print(f"  ROC-AUC: {results_1h.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_1h.get('clf_CSI', 'N/A'):.4f}")


## Pipeline Prediksi 3 Jam
Menjalankan pipeline lengkap LSTM untuk prediksi curah hujan 3 jam ke depan.

In [ ]:
results_3h = run_lstm_pipeline('3h', df_feat)
print("\n[SELESAI] Pipeline LSTM 3 Jam")
if results_3h:
    print(f"  ROC-AUC: {results_3h.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_3h.get('clf_CSI', 'N/A'):.4f}")


## Pipeline Prediksi Harian
Menjalankan pipeline lengkap LSTM untuk prediksi curah hujan harian (24 jam) ke depan.

In [ ]:
results_daily = run_lstm_pipeline('daily', df_feat)
print("\n[SELESAI] Pipeline LSTM Harian")
if results_daily:
    print(f"  ROC-AUC: {results_daily.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_daily.get('clf_CSI', 'N/A'):.4f}")


## Laporan Perbandingan Akhir
Membandingkan kinerja LSTM vs XGBoost untuk semua skala waktu dan mengekspor laporan ringkasan.

In [ ]:
def generate_comparison_report(results_list, output_base):
    valid = [r for r in results_list if r is not None]
    if not valid:
        logger.warning("Tidak ada hasil untuk dibandingkan")
        return None

    df_cmp = pd.DataFrame(valid)
    clf_cols = ['clf_ROC_AUC','clf_F1','clf_CSI','clf_POD','clf_FAR','clf_ETS','clf_HSS',
                'clf_Accuracy','clf_Precision','clf_Recall','clf_PR_AUC','clf_Brier_Cal']
    reg_cols = ['reg_RMSE','reg_MAE','reg_R2','reg_NSE','reg_KGE','reg_Bias','reg_Correlation']
    all_cols = ['model','scale'] + [c for c in clf_cols + reg_cols if c in df_cmp.columns]
    df_export = df_cmp[all_cols].copy()

    path_csv = output_base.parent / 'model_comparison_lstm.csv'
    df_export.to_csv(path_csv, index=False)
    logger.info(f"Laporan perbandingan LSTM disimpan: {path_csv}")

    # Gabungkan dengan XGBoost jika ada
    xgb_path = output_base.parent / 'model_comparison_xgboost.csv'
    if xgb_path.exists():
        df_xgb = pd.read_csv(xgb_path)
        df_all = pd.concat([df_export, df_xgb], ignore_index=True)
        path_all = output_base.parent / 'model_comparison.csv'
        df_all.to_csv(path_all, index=False)
        logger.info(f"Laporan perbandingan gabungan disimpan: {path_all}")
    else:
        df_all = df_export

    # Plot ringkasan perbandingan
    metrics_to_plot = [
        ('clf_ROC_AUC', 'ROC-AUC'),
        ('clf_CSI', 'CSI'),
        ('reg_RMSE', 'RMSE (mm)'),
    ]
    scales_ordered = ['1h', '3h', 'daily']
    scale_labels   = ['1 Jam', '3 Jam', 'Harian']

    n_plots = len(metrics_to_plot)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    for ax, (metric, label) in zip(axes, metrics_to_plot):
        models_in_data = df_all['model'].unique()
        x = np.arange(len(scales_ordered))
        width = 0.35
        for mi, mdl in enumerate(models_in_data):
            vals = []
            for sc in scales_ordered:
                row = df_all[(df_all['model']==mdl) & (df_all['scale']==sc)]
                val = row[metric].values[0] if len(row) > 0 and metric in row.columns and not pd.isna(row[metric].values[0]) else 0
                vals.append(val)
            offset = (mi - len(models_in_data)/2 + 0.5) * width
            bars = ax.bar(x + offset, vals, width, label=mdl.upper())
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=7)

        ax.set_xticks(x)
        ax.set_xticklabels(scale_labels)
        ax.set_title(label, fontsize=13, fontweight='bold')
        ax.set_ylabel(label)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle('Perbandingan Kinerja LSTM vs XGBoost — Multi-Skala Waktu', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path_fig = output_base.parent / 'summary_comparison_all_models.png'
    fig.savefig(path_fig, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info(f"Ringkasan plot disimpan: {path_fig}")

    display(df_all if xgb_path.exists() else df_export)
    return df_all if xgb_path.exists() else df_export

# Buat laporan perbandingan
all_results = [results_1h, results_3h, results_daily]
df_comparison = generate_comparison_report(all_results, OUTPUT_BASE)
print("\n=== PIPELINE LSTM V2 SELESAI ===")
print(f"Output tersimpan di: {OUTPUT_BASE}")
